# tensor-wraps-ndarray — ex1: compare from_numpy aliasing vs tensor copy

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-wraps-ndarray`. Running the final beacon cell reports progress against the `PyTorch: tensor from ndarray` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: tensor from ndarray` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-wraps-ndarray`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-wraps-ndarray"
DD_SUBTOPIC = "PyTorch: tensor from ndarray"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## tensor from ndarray — quick refresher

Three ways to make a torch tensor from a NumPy ndarray; they have **different semantics**.

| factory | shares memory? | dtype-preserving? |
|---|---|---|
| `t.from_numpy(arr)` | YES — zero-copy view | yes |
| `t.as_tensor(arr)` | yes when possible (same dtype+device) | yes |
| `t.tensor(arr)` | NO — always copies | yes (deduced) |

**Aliasing trap.** Tensors built with `from_numpy` write through to the source ndarray. Mutating the tensor in place will silently change the NumPy view — and vice versa.

**Dtype gotcha.** NumPy defaults to `float64`; PyTorch defaults to `float32`. Cast at construction (`t.from_numpy(arr).float()` or `arr.astype('float32')`) to avoid an unexpected double-precision tensor.

### Exercise 1 — compare from_numpy aliasing vs tensor copy

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Analyze
> LO: Distinguish `t.from_numpy` (shares memory with the source ndarray) from `t.tensor` (copies) by mutating the source and observing which torch tensor reflects the change.
> Keywords: from_numpy, tensor-factory, aliasing, copy
> ```

**KCs targeted:** `from-numpy-shares-storage`, `tensor-factory-copies`

Implement `ex1_aliasing_report(arr)`.

Given a NumPy `ndarray` `arr`, build:
1. `wrapped = t.from_numpy(arr)` — should share memory with `arr`.
2. `copied = t.tensor(arr)` — should be an independent copy.

Then mutate `arr` in place (set `arr[0] = 999`) and read the first element of each tensor. Return a dict:
```python
{
    'wrapped_first': float(wrapped[0]),
    'copied_first':  float(copied[0]),
    'wrapped_shares_storage': <bool — wrapped[0] tracked the mutation>,
    'copied_shares_storage':  <bool — copied[0] tracked the mutation>,
}
```

The test will pre-record the original first-element value and verify your aliasing predictions are correct.

Input: `arr` — `np.ndarray`, dtype `float32`, length ≥ 1.
Output: dict with the four keys above.

In [ ]:
def ex1_aliasing_report(arr) -> dict:
    """Build a from_numpy tensor and a t.tensor copy; mutate arr; report aliasing."""
    raise NotImplementedError()


def _test_ex1():
    arr = np.arange(5, dtype=np.float32)
    # arr[0] starts at 0.0; the function should mutate it to 999.0.
    report = ex1_aliasing_report(arr)

    assert isinstance(report, dict), 'must return a dict'
    for key in ('wrapped_first', 'copied_first',
                'wrapped_shares_storage', 'copied_shares_storage'):
        assert key in report, f'missing key {key!r}'

    # from_numpy aliases storage → wrapped saw the 999.
    assert report['wrapped_first'] == 999.0, (
        f'wrapped_first should be 999.0 (aliasing), got {report["wrapped_first"]}'
    )
    assert report['wrapped_shares_storage'] is True, (
        'from_numpy MUST share storage with the source ndarray'
    )

    # t.tensor copies → copied did NOT see the 999.
    assert report['copied_first'] == 0.0, (
        f'copied_first should be 0.0 (independent copy), got {report["copied_first"]}'
    )
    assert report['copied_shares_storage'] is False, (
        't.tensor MUST NOT share storage with the source ndarray'
    )

    # Confirm the function actually mutated arr (not just claimed to).
    assert arr[0] == 999.0, 'function should have mutated arr[0] in place'
    print('from_numpy aliases; t.tensor copies — both verified by mutation test')
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_aliasing_report(arr) -> dict:
    wrapped = t.from_numpy(arr)
    copied = t.tensor(arr)
    original = float(arr[0])
    arr[0] = 999.0
    return {
        'wrapped_first': float(wrapped[0]),
        'copied_first':  float(copied[0]),
        'wrapped_shares_storage': float(wrapped[0]) != original,
        'copied_shares_storage':  float(copied[0]) != original,
    }
```

**`t.from_numpy` is a zero-copy wrap.** The torch tensor and the source ndarray share the same memory buffer. Mutate either, the other sees it. Useful for fast NumPy → torch in a data pipeline where you don't want to double-allocate.

**`t.tensor(arr)` always copies.** This is the safe-but-slower factory. If you're at all unsure about who else might mutate the source array, prefer this over `from_numpy`.

**`t.as_tensor(arr)` is the middle ground.** Shares memory when possible (matching dtype and device, source is an ndarray), copies otherwise. Good default for generic 'turn this into a tensor' code paths.

**Dtype trap.** NumPy's default float dtype is `float64`. If you `t.from_numpy(np.array([1.0, 2.0]))` you get a `float64` torch tensor, which will misbehave the moment it touches a `float32` model. Either build the ndarray with `dtype='float32'` from the start, or chain `.float()` on the torch tensor.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()